# Module 3 Simple Code Along

Goal: understand how classes, type hints, decorators, and FastAPI fit together.

Core idea:

> Classes organize behavior. Decorators attach behavior. FastAPI exposes behavior over HTTP.

## 1. A simple class

A class groups data and behavior together.

In [ ]:
class Product:
    def __init__(self, id, name, price):
        self.id = id
        self.name = name
        self.price = price

    def label(self):
        return f"{self.name} - Rs {self.price}"


product = Product(1, "Cable", 499)

print(product.name)
print(product.label())

## 2. Add a rule to the class

A class is useful because it can protect rules. Here, price cannot be negative.

In [ ]:
class Product:
    def __init__(self, id, name, price):
        if price < 0:
            raise ValueError("price must be non-negative")
        self.id = id
        self.name = name
        self.price = price

    def label(self):
        return f"{self.name} - Rs {self.price}"


good = Product(1, "Cable", 499)
print(good.label())

# Uncomment this line to see the rule fail:
# bad = Product(2, "Broken Item", -50)

## 3. A catalog class

The catalog stores products in a dict internally, but users call methods.

In [ ]:
class ProductCatalog:
    def __init__(self):
        self._items = {}

    def add(self, product):
        if product.id in self._items:
            raise ValueError("duplicate product id")
        self._items[product.id] = product

    def get(self, product_id):
        return self._items[product_id]

    def list_all(self):
        return list(self._items.values())


catalog = ProductCatalog()
catalog.add(Product(1, "Cable", 499))
catalog.add(Product(2, "Keyboard", 5499))

print(catalog.get(2).name)
print([p.name for p in catalog.list_all()])

## 4. Dataclass: less boilerplate

`@dataclass` creates `__init__`, `__repr__`, and equality for simple data classes.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class ProductData:
    id: int
    name: str
    price: float
    tags: list[str] = field(default_factory=list)


p1 = ProductData(1, "Cable", 499)
p2 = ProductData(1, "Cable", 499)

print(p1)
print(p1 == p2)
print(p1.tags)

Important: dataclasses make classes easier to write, but they do not validate types at runtime. Pydantic does that in Day 2.

## 5. Type hints

Type hints help readers, editors, FastAPI, Pydantic, and tests.

In [ ]:
def price_with_tax(price: float) -> float:
    return round(price * 1.18, 2)


print(price_with_tax(100))
print(price_with_tax.__annotations__)

## 6. A function can receive another function

In Python, a function is a value. You can pass it into another function.

In [ ]:
def say_hello():
    print("hello")


def run_this(func):
    print("before")
    func()
    print("after")


run_this(say_hello)

## 7. A simple decorator

A decorator takes a function, wraps it with extra behavior, and returns the wrapper.

In [ ]:
def log_calls(func):
    def wrapper():
        print("calling", func.__name__)
        func()
        print("finished", func.__name__)
    return wrapper


@log_calls
def greet():
    print("hello from greet")


greet()

The `@log_calls` line means this:

```python
greet = log_calls(greet)
```

That is the whole decorator idea.

In [ ]:
# Most real decorators use *args and **kwargs so they work with any function shape.

def log_any_call(func):
    def wrapper(*args, **kwargs):
        print("calling", func.__name__)
        return func(*args, **kwargs)
    return wrapper


@log_any_call
def add(a, b):
    return a + b


print(add(2, 3))

## 8. Preserve function names with `functools.wraps`

Without `functools.wraps`, Python thinks the decorated function is named `wrapper`.

In [ ]:
import functools


def log_without_wraps(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@log_without_wraps
def original_function():
    return "done"


print(original_function())
print(original_function.__name__)

Now use `functools.wraps(func)` to keep the original function's name and metadata.

In [ ]:
def log_with_wraps(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@log_with_wraps
def original_function_again():
    return "done"


print(original_function_again())
print(original_function_again.__name__)

## 9. Optional: a tiny retry decorator

`retry` is a decorator with configuration. This is useful when something can fail temporarily, like a network call.

In [ ]:
def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper():
            for attempt in range(1, times + 1):
                try:
                    print("attempt", attempt)
                    return func()
                except Exception:
                    if attempt == times:
                        raise
        return wrapper
    return decorator


calls = {"count": 0}


@retry(3)
def sometimes_fails():
    calls["count"] += 1
    if calls["count"] < 3:
        raise ValueError("temporary problem")
    print("success")


sometimes_fails()

## 10. FastAPI is the same decorator idea

FastAPI uses decorators to register Python functions as API routes.

We will not run a real server inside this notebook. This tiny example shows the idea.

In [ ]:
routes = {}


def get(path):
    def decorator(func):
        routes[path] = func
        return func
    return decorator


@get("/health")
def health():
    return {"status": "ok"}


@get("/products")
def list_products():
    return [
        {"id": 1, "name": "Cable"},
        {"id": 2, "name": "Keyboard"},
    ]


print(routes)
print(routes["/health"]())
print(routes["/products"]())

Real FastAPI looks like this:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/health")
def health() -> dict:
    return {"status": "ok"}
```

Then run:

```bash
uvicorn catalog.server:app --reload
```

And open:

```text
http://localhost:8000/docs
```

## 11. Mini recap

Use this mental model:

```text
class
    organizes state and behavior

type hints
    describe data for people and tools

decorator
    wraps or registers a function

FastAPI
    uses decorators + type hints to expose Python functions as HTTP routes
```

Short version:

- `ProductCatalog` owns the business behavior.
- `server.py` exposes that behavior as an API.
- `@retry` wraps functions with resilience.
- `@app.get` registers functions as routes.
- The same decorator pattern returns later in pytest and LLM tools.